In [ ]:
import os
import json
import re
import torch
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "artifacts").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not locate the MIRx project root from {start}")

PROJECT_ROOT = find_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
TOP_K = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
embeddings = np.load(ARTIFACTS_DIR / "embeddings.npy")

metadata = pd.read_parquet(ARTIFACTS_DIR / "metadata.parquet")
if "document_id" not in metadata.columns:
    metadata = metadata.copy()
    metadata["document_id"] = "doc_" + pd.Series(np.arange(len(metadata))).astype(str).str.zfill(6)

config = json.load(open(ARTIFACTS_DIR / "embedding_config.json", "r"))

print("Embeddings:", embeddings.shape)
print("Metadata:", metadata.shape)
print("Embedding model:", config["embedding_model"])

In [ ]:
embedding_model = SentenceTransformer(config["embedding_model"], device = DEVICE)
print("Embedding model loaded:", embedding_model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype = "auto", device_map = "auto")
llm_model.eval()

In [ ]:
def retrieve(query, top_k = 5):
    query_embedding = embedding_model.encode([query], convert_to_numpy = True, normalize_embeddings = True)
    scores = cosine_similarity(query_embedding, embeddings)[0]

    top_k = np.argsort(scores)[::-1][:top_k]
    results = metadata.iloc[top_k].copy()

    results["similarity"] = scores[top_k]

    return results.reset_index(drop = True)

In [ ]:
TEXT_COLUMN = "document"

if TEXT_COLUMN not in metadata.columns:
    raise ValueError(
        f"Expected '{TEXT_COLUMN}' column, "
        f"but found: {metadata.columns.tolist()}"
    )

In [ ]:
def build_context(results):

    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"""
SOURCE {i + 1}
Similarity: {row["similarity"]:.4f}

{row[TEXT_COLUMN]}
"""
        )

    return "\n".join(context_parts)

In [ ]:
SYSTEM_PROMPT = """
You are MIRx, a medical information retrieval assistant.

Your task is to answer questions using ONLY the information
provided in the retrieved context.

Rules:

1. Do not invent information.
2. Do not introduce medical facts that are absent from the context.
3. If the retrieved context is insufficient, say so clearly.
4. Keep the answer concise and well structured.
5. When possible, mention the relevant drug or medical entity.
6. Do not make diagnoses or provide personalized medical treatment.
7. Clearly distinguish information found in the retrieved context
   from general uncertainty.
"""

In [ ]:
def generate_with_qwen(query, context):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Retrieved medical information:

{context}

User question:

{query}

Using only the retrieved medical information above,
answer the user's question.
"""
        }
    ]

    # Create model inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move inputs to the model device
    inputs = inputs.to(
        next(llm_model.parameters()).device
    )

    # Number of input tokens
    input_length = inputs["input_ids"].shape[-1]

    # Generate
    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    # Extract only newly generated tokens
    generated_tokens = outputs[
        0,
        input_length:
    ]

    # Decode
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
def generate_response(query, top_k = 5):

    # -------------------------
    # 1. Retrieve
    # -------------------------

    results = retrieve(query, top_k = top_k)

    # -------------------------
    # 2. Build context
    # -------------------------

    context = build_context(results)

    # -------------------------
    # 3. Generate answer
    # -------------------------

    answer = generate_with_qwen(query, context)

    return answer, results

In [ ]:
query = "What are the alternatives to paracetamol?"

answer, sources = generate_response(query, top_k = 5)

print("=" * 80)
print("MIRx RESPONSE")
print("=" * 80)

print(answer)

print("\n")
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, row in sources.iterrows():

    print(
        f"\n[{i + 1}] "
        f"Similarity: {row['similarity']:.4f}"
    )

    print(row["document"])

In [ ]:
print(metadata.columns.tolist())

In [ ]:
display(metadata.head())

In [ ]:
test_queries = [
    "What are the alternatives to paracetamol?",
    "What are the side effects of ibuprofen?",
    "Which drugs are used for pain relief?",
    "What are the alternatives to aspirin?",
    "Which medicines are associated with nausea?"
]

for query in test_queries:

    print("\n")
    print("#" * 100)
    print("QUERY:", query)

    answer, sources = generate_response(
        query,
        top_k=5
    )

    print("\nANSWER:")
    print(answer)

# LLM Generation Evaluation

This evaluation block uses the actual MIRx generation pipeline above. It loads the same ground-truth/test-query files used by retrieval evaluation and evaluates the generated answer against the retrieved context and gold answer. Each generation metric is isolated in its own cell.

**PoC note:** the notebook uses the loaded Qwen model as the judge for semantic metrics. For the final research paper, use a fixed independent judge model and report the judge model/version and temperature.

In [ ]:
# ============================================================
# EVALUATION CONFIGURATION AND DATA
# ============================================================

GROUND_TRUTH_DIR = ARTIFACTS_DIR / "evaluation" / "ground_truth.json"
TEST_QUERIES_DIR = ARTIFACTS_DIR / "evaluation" / "test_queries.json"
GENERATION_RESULTS_PATH = DATA_DIR / "evaluation" / "poc_generation_results.json"

if GROUND_TRUTH_DIR.exists():
    with open(GROUND_TRUTH_DIR, "r", encoding="utf-8") as f:
        ground_truth = json.load(f)
else:
    ground_truth = {"queries": []}
    print("Generation evaluation skipped: no finalized ground_truth.json. Complete human annotation in 005_Ground_Truth_Population_FINAL.ipynb first.")

with open(TEST_QUERIES_DIR, "r", encoding="utf-8") as f:
    test_queries_raw = json.load(f)

if isinstance(ground_truth, dict):
    gt_records_raw = ground_truth.get("queries", ground_truth.get("records", []))
elif isinstance(ground_truth, list):
    gt_records_raw = ground_truth
else:
    raise TypeError("Unsupported ground-truth JSON format.")

gt_records = {
    str(r["query_id"]): r
    for r in gt_records_raw
    if isinstance(r, dict) and "query_id" in r
}

if isinstance(test_queries_raw, dict):
    test_queries_raw = (
        test_queries_raw.get("queries")
        or test_queries_raw.get("test_queries")
        or test_queries_raw.get("records")
        or []
    )
elif not isinstance(test_queries_raw, list):
    raise TypeError("Unsupported test-query JSON format.")

def norm_text(text):
    return " ".join(str(text).lower().split())

gt_text_lookup = {}
for qid, record in gt_records.items():
    text = record.get("query") or record.get("question") or record.get("query_text")
    if text:
        gt_text_lookup[norm_text(text)] = qid

normalized_test_queries = []
for i, item in enumerate(test_queries_raw):
    if isinstance(item, str):
        query = item
        qid = gt_text_lookup.get(norm_text(query), f"MIRX-Q-{i + 1:04d}")
    elif isinstance(item, dict):
        query = item.get("query") or item.get("question") or item.get("query_text") or item.get("text")
        if not query:
            continue
        qid = item.get("query_id") or gt_text_lookup.get(norm_text(query), f"MIRX-Q-{i + 1:04d}")
    else:
        continue
    normalized_test_queries.append({"query_id": str(qid), "query": str(query)})

matched = [q for q in normalized_test_queries if q["query_id"] in gt_records]

print("Ground-truth records:", len(gt_records))
print("Test queries:", len(normalized_test_queries))
print("Matched queries:", len(matched))

if gt_records and not matched:
    raise ValueError("No test-query IDs match the ground truth.")


In [ ]:
# ============================================================
# GENERATION OPERATION — RUN ACTUAL MIRx PIPELINE
# ============================================================

import time

generation_records = []

for item in matched:
    query_id = item["query_id"]
    query = item["query"]

    start = time.perf_counter()

    answer, sources = generate_response(
        query,
        top_k=TOP_K
    )

    latency_ms = (time.perf_counter() - start) * 1000

    source_records = []
    for rank, (_, row) in enumerate(sources.iterrows(), start=1):
        source_records.append({
            "rank": rank,
            "chunk_id": str(row.get("chunk_id", row.get("document_id", row.get("id", rank)))),
            "similarity": float(row["similarity"]),
            "text": str(row[TEXT_COLUMN])
        })

    generation_records.append({
        "query_id": query_id,
        "query": query,
        "answer": answer,
        "retrieved_chunks": source_records,
        "latency_ms": latency_ms
    })

# Save the raw generation trace so the experiment is reproducible.
GENERATION_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(GENERATION_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(generation_records, f, ensure_ascii=False, indent=2)

print("Generation completed:", len(generation_records))
print("Saved to:", GENERATION_RESULTS_PATH)


In [ ]:
# ============================================================
# LLM JUDGE HELPER
# ============================================================

def run_judge(prompt):
    messages = [
        {
            "role": "system",
            "content": "You are a strict evaluator of medical RAG answers. Return ONLY valid JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(next(llm_model.parameters()).device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated = outputs[0, inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()

    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        raise ValueError(f"Judge did not return JSON: {response}")

    return json.loads(match.group(0))


In [ ]:
# ============================================================
# 1. Faithfulness
# ============================================================

def faithfulness_metric(query, answer, context):
    prompt = f"""QUESTION:\n{query}\n\nRETRIEVED CONTEXT:\n{context}\n\nANSWER:\n{answer}\n\nScore whether factual claims in the answer are supported by the retrieved context. Return JSON: {{\"score\": 0.0, \"unsupported_claims\": []}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 2. Answer Correctness
# ============================================================

def answer_correctness_metric(query, answer, gold_answer, key_facts):
    prompt = f"""QUESTION:\n{query}\n\nGOLD ANSWER:\n{gold_answer}\n\nKEY FACTS:\n{json.dumps(key_facts, ensure_ascii=False)}\n\nGENERATED ANSWER:\n{answer}\n\nScore factual correctness. Return JSON: {{\"score\": 0.0, \"incorrect_claims\": []}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 3. Answer Relevance
# ============================================================

def answer_relevance_metric(query, answer):
    prompt = f"""QUESTION:\n{query}\n\nANSWER:\n{answer}\n\nScore how directly the answer addresses the question. Return JSON: {{\"score\": 0.0}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 4. Completeness
# ============================================================

def completeness_metric(answer, key_facts, must_include):
    required = list(dict.fromkeys(key_facts + must_include))
    if not required:
        return np.nan
    prompt = f"""REQUIRED FACTS:\n{json.dumps(required, ensure_ascii=False)}\n\nANSWER:\n{answer}\n\nScore the fraction and correctness of required information addressed. Return JSON: {{\"score\": 0.0, \"missing_facts\": []}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 5. Hallucination Rate
# ============================================================

def hallucination_rate_metric(answer, context, must_not_claim):
    prompt = f"""CONTEXT:\n{context}\n\nANSWER:\n{answer}\n\nPROHIBITED CLAIMS:\n{json.dumps(must_not_claim, ensure_ascii=False)}\n\nEstimate the proportion of factual claims that are unsupported by context or violate prohibited claims. Return JSON: {{\"hallucination_rate\": 0.0, \"hallucinated_claims\": []}}. Score 0-1."""
    return float(run_judge(prompt)["hallucination_rate"])


In [ ]:
# ============================================================
# 6. Citation / Evidence Correctness
# ============================================================

def citation_evidence_correctness_metric(answer, sources, gold_evidence):
    prompt = f"""ANSWER:\n{answer}\n\nRETRIEVED SOURCES:\n{json.dumps(sources, ensure_ascii=False)}\n\nGOLD EVIDENCE:\n{json.dumps(gold_evidence, ensure_ascii=False)}\n\nScore whether retrieved evidence supports the answer. Return JSON: {{\"score\": 0.0, \"unsupported_evidence\": []}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 7. Response Consistency
# ============================================================

def response_consistency_metric(answers):
    # A single deterministic run cannot establish consistency.
    # This metric becomes active when repeated runs are collected.
    if len(answers) < 2:
        return np.nan
    prompt = f"""ANSWERS TO THE SAME QUESTION:\n{json.dumps(answers, ensure_ascii=False)}\n\nScore factual consistency across answers. Return JSON: {{\"score\": 0.0}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ============================================================
# 8. LLM-as-a-Judge — Overall Quality
# ============================================================

def llm_judge_metric(query, answer, context, gold_answer):
    prompt = f"""QUESTION:\n{query}\n\nGOLD ANSWER:\n{gold_answer}\n\nCONTEXT:\n{context}\n\nANSWER:\n{answer}\n\nScore overall quality considering correctness, relevance, completeness, faithfulness, clarity and medical safety. Do not reward unsupported claims. Return JSON: {{\"score\": 0.0}}. Score 0-1."""
    return float(run_judge(prompt)["score"])


In [ ]:
# ===================================
# RUN ALL GENERATION METRICS 
# ===================================

generation_evaluation_rows = []

for record in generation_records:

    qid = record["query_id"]

    if qid not in gt_records:
        print(f"[SKIP] No ground truth for {qid}")
        continue

    gt = gt_records[qid]

    answer = record.get("answer", "")
    sources = record.get("retrieved_chunks", [])

    context = "\n\n".join(
        s.get("text", "")
        for s in sources
        if isinstance(s, dict)
    )

    gold = gt.get("gold_answer", {})

    gold_answer = (
        gold.get("canonical", "")
        if isinstance(gold, dict)
        else str(gold)
    )

    key_facts = (
        gold.get("key_facts", [])
        if isinstance(gold, dict)
        else []
    )

    must_include = (
        gold.get("must_include", [])
        if isinstance(gold, dict)
        else []
    )

    must_not_claim = (
        gold.get("must_not_claim", [])
        if isinstance(gold, dict)
        else []
    )

    gold_evidence = gt.get(
        "gold_evidence",
        []
    )

    row = {
        "query_id": qid,
        "query": record.get("query", ""),
    }

    # --------------------------------------------------------
    # Faithfulness
    # --------------------------------------------------------

    try:

        row["Faithfulness"] = faithfulness_metric(
            record["query"],
            answer,
            context,
        )

    except Exception as e:

        print(
            f"[ERROR] Faithfulness | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["Faithfulness"] = np.nan


    # --------------------------------------------------------
    # Answer Correctness
    # --------------------------------------------------------

    try:

        row["AnswerCorrectness"] = (
            answer_correctness_metric(
                record["query"],
                answer,
                gold_answer,
                key_facts,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] AnswerCorrectness | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["AnswerCorrectness"] = np.nan


    # --------------------------------------------------------
    # Answer Relevance
    # --------------------------------------------------------

    try:

        row["AnswerRelevance"] = (
            answer_relevance_metric(
                record["query"],
                answer,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] AnswerRelevance | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["AnswerRelevance"] = np.nan


    # --------------------------------------------------------
    # Completeness
    # --------------------------------------------------------

    try:

        row["Completeness"] = (
            completeness_metric(
                answer,
                key_facts,
                must_include,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] Completeness | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["Completeness"] = np.nan


    # --------------------------------------------------------
    # Hallucination Rate
    # --------------------------------------------------------

    try:

        row["HallucinationRate"] = (
            hallucination_rate_metric(
                answer,
                context,
                must_not_claim,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] HallucinationRate | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["HallucinationRate"] = np.nan


    # --------------------------------------------------------
    # Citation Evidence Correctness
    # --------------------------------------------------------

    try:

        row["CitationEvidenceCorrectness"] = (
            citation_evidence_correctness_metric(
                answer,
                sources,
                gold_evidence,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] CitationEvidenceCorrectness | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["CitationEvidenceCorrectness"] = np.nan


    # --------------------------------------------------------
    # Response Consistency
    # --------------------------------------------------------

    row["ResponseConsistency"] = np.nan


    # --------------------------------------------------------
    # LLM Judge
    # --------------------------------------------------------

    try:

        row["LLMJudge"] = (
            llm_judge_metric(
                record["query"],
                answer,
                context,
                gold_answer,
            )
        )

    except Exception as e:

        print(
            f"[ERROR] LLMJudge | "
            f"{qid} | {type(e).__name__}: {e}"
        )

        row["LLMJudge"] = np.nan


    # --------------------------------------------------------
    # Latency
    # --------------------------------------------------------

    row["Latency_ms"] = record.get(
        "latency_ms",
        np.nan,
    )

    generation_evaluation_rows.append(row)


generation_metrics_df = pd.DataFrame(
    generation_evaluation_rows
)


print(
    f"\nGeneration evaluations produced: "
    f"{len(generation_metrics_df)}"
)

display(
    generation_metrics_df.head()
)

In [ ]:
# ============================================================
# GENERATION METRICS — AGGREGATE RESULTS
# ============================================================

metric_names = [
    "Faithfulness",
    "AnswerCorrectness",
    "AnswerRelevance",
    "Completeness",
    "HallucinationRate",
    "CitationEvidenceCorrectness",
    "ResponseConsistency",
    "LLMJudge"
]

summary_rows = []
for metric in metric_names:
    values = generation_metrics_df[metric].dropna()
    summary_rows.append({
        "Metric": metric,
        "Mean": values.mean(),
        "Std": values.std(),
        "N": len(values)
    })

generation_summary_df = pd.DataFrame(summary_rows)
generation_summary_df[["Mean", "Std"]] = generation_summary_df[["Mean", "Std"]].round(4)

print("=" * 70)
print("MIRx — LLM GENERATION RESULTS")
print("=" * 70)
display(generation_summary_df)
